### Members' Full name:

- Nguyen Mai Dinh, Le - 300312139
- Ruiz, Ricardo - 300387021
- Jimmy, Suwarly - 300361475
- Hugh, Tran - 300394597

# Part A. Planning

# Part B. Basic Model

### 1. Import Library + Data Loading (Demi)

In [9]:
import pandas as pd
import numpy as np

In [10]:
df1 = pd.read_csv("faq.csv")
df2 = pd.read_csv("faq1.csv")
df3 = pd.read_csv("faq2.csv")
df4 = pd.read_csv("faq3.csv")
df5 = pd.read_csv("faq4.csv")
df6 = pd.read_csv("faq5.csv")
df7 = pd.read_csv("faq6.csv")
df8 = pd.read_csv("faq7.csv")

df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8], ignore_index=True)
df

,question,answer
0,How many programs can I apply to?,You can indicate one program choice per applic...
1,What is the fee to apply?,$37.40 for Canadian/Permanent Residents. $100 ...
2,What is preferential admission?,Preferential admission is a selective admissio...
3,How long is the wait-list for Douglas College ...,There is no wait-list for open enrolment or li...
4,I got a letter saying I'm in the program. Why ...,The first letter you received was an acknowled...
...,...,...
72,"If a student has to stay home (flu symptoms, f...",It is expected that we may have higher than no...
73,"If I am sick, are there alternatives to attend...",Decision regarding attendance is the discretio...
74,Will there be more options for online learning...,While Douglas College is returning to primaril...
75,Will I be able to access online classes / reco...,Not necessarily. Students should not have the ...


### 2. Search Engine Building (Ricardo)

### 3. Testing + Evaluation

#### 3.1. Precision@K Function + Testing + Evaluation function (Demi)

In [11]:
def run_queries(es, index_name, queries, top_k=5):
    results = {}
    
    for q in queries:
        response = es.search(
            index=index_name,
            body={
                "query": {
                    "multi_match": {
                        "query": q,
                        "fields": ["question^2", "answer"]
                    }
                },
                "size": top_k
            }
        )
        
        docs = []
        for hit in response["hits"]["hits"]:
            docs.append(hit["_source"])
        
        results[q] = docs
    
    return results

def precision_at_k(relevance_list, k=1):
    return sum(relevance_list[:k]) / k

def evaluate_precision_at_1(relevance_judgments):
    scores = []
    
    for rels in relevance_judgments.values():
        p1 = precision_at_k(rels, k=1)
        scores.append(p1)
    
    avg_p1 = sum(scores) / len(scores)
    
    return scores, avg_p1

def evaluate_precision_at_k(relevance_judgments, k=5):
    scores = []
    
    for rels in relevance_judgments.values():
        pk = precision_at_k(rels, k=k)
        scores.append(pk)
    
    avg_pk = sum(scores) / len(scores)
    
    return scores, avg_pk

#### 3.2. 20 questions + Evaluation (All)

**Demi**

5 questions

Testing using the pre-built testing function

Evaluation

**Ricardo**

5 questions

Testing using the pre-built testing function

Evaluation

**Jimmy**

5 questions

Testing using the pre-built testing function

Evaluation

**Hugh**

5 questions:
- How can I submit my application?
- What do I need to pay for tuition?
- Where can I get advising help?
- How do I talk to a student advisor?
- What happens if I fail a course?

Testing using the pre-built testing function

Evaluation

# Part C. ElasticSearch with mbedding

### 1. Discussion (Demi)

### 2. Building Embedding (Hugh)

In [12]:
# pip install elasticsearch
# %pip install -U spacy
# !python -m spacy download en_core_web_lg

import re
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from elasticsearch import Elasticsearch
# warnings.filterwarnings("ignore")

es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", "3y-I0U=GUbfXRiTe+*Rn"),
    verify_certs=False
)
print(es.info())

{'name': 'HTTK', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'MlUnGcu4RrWRF8cRp--4lg', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


C:\Users\qhung\anaconda3\Lib\site-packages\elasticsearch\_sync\client\__init__.py:313: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
C:\Users\qhung\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [13]:
nlp = spacy.load("en_core_web_lg")

In [14]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["question_clean"] = df["question"].apply(clean_text)
df["answer_clean"] = df["answer"].apply(clean_text)
# df.info()

In [15]:
def get_embedding(text, nlp_model):
    return nlp_model(text).vector

df["question_vector"] = df["question_clean"].apply(lambda x: get_embedding(x, nlp))
# df["question_vector"]

In [16]:
faq_matrix = np.vstack(df["question_vector"].values)
faq_matrix.shape

(77, 300)

In [17]:
def semantic_search(query, df, faq_matrix, nlp_model, top_k=5):
    query_clean = clean_text(query)
    query_vector = nlp_model(query_clean).vector.reshape(1, -1)

    similarities = cosine_similarity(query_vector, faq_matrix)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]

    return results[["question", "answer", "similarity_score"]]

In [18]:
test_query = "How can I apply to Douglas?"
semantic_search(test_query, df, faq_matrix, nlp, top_k=5)

,question,answer,similarity_score
0,How many programs can I apply to?,You can indicate one program choice per applic...,0.929358
6,What if I don't get in this year? Do I have to...,If all admission requirements have been met fo...,0.904084
49,I am almost done my program. How do I apply to...,You should apply to graduate after you have en...,0.901134
12,"I am out of the country, when should I come to...",Classes that are scheduled as “in-person” or “...,0.897031
61,How quickly can I get an appointment?,Counselling appointments are typically booked ...,0.894558


In [20]:
def run_semantic_queries(df, faq_matrix, queries, nlp_model, top_k=5):
    results = {}

    for q in queries:
        result_df = semantic_search(q, df, faq_matrix, nlp_model, top_k=top_k)
        results[q] = result_df

    return results

### 2. Testing + Evaluation (All)

In [24]:
hugh_test_questions = [
    "How can I submit my application?",
    "What do I need to pay for tuition?",
    "Where can I get advising help?",
    "How do I talk to a student advisor?",
    "What happens if I fail a course?"
]

In [26]:
semantic_results = run_semantic_queries(df, faq_matrix, hugh_test_questions, nlp, top_k=5)

for q, res in semantic_results.items():
    print("=" * 100)
    print("QUESTION:", q)
    print(res)
    print()

QUESTION: How can I submit my application?
                                             question  \
4   I got a letter saying I'm in the program. Why ...   
49  I am almost done my program. How do I apply to...   
61              How quickly can I get an appointment?   
0                   How many programs can I apply to?   
62  Do I need to prepare anything for my appointment?   

                                               answer  similarity_score  
4   The first letter you received was an acknowled...          0.932356  
49  You should apply to graduate after you have en...          0.920658  
61  Counselling appointments are typically booked ...          0.913821  
0   You can indicate one program choice per applic...          0.913310  
62  No, you do not need to prepare anything. Howev...          0.909747  

QUESTION: What do I need to pay for tuition?
                                             question  \
62  Do I need to prepare anything for my appointment?   
6   What i

# Part D. FAQBot using Gemini

### 1. RAG Building (Jimmy)

### 2. Testing + Evaluation (All)

# Part E. Final Conclusion + Compare Results (Demi)